# Assignment 1 — Gradio dashboard

You're using a notebook for this stage. If you'd rather use a script,
write `app.py` instead and delete this file — `main.py` refuses to run if
it finds both `app.py` and `app.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py app`, this notebook must build and launch a Gradio
app (a `demo` that calls `.launch()`) using only what your training stage
already produced — it must never retrain anything itself. At minimum, the
dashboard must let you:

- Compare your three trained models via a prediction-vs-actual plot.
- See feature and/or target distributions.
- For classification: move a decision-threshold slider and watch the
  confusion matrix, and a business-cost number tied to your `REPORT.md`'s
  framing, change with it.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

In [ ]:
import json

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn

ART = "artifacts"
MODEL_LABELS = {"sklearn": "scikit-learn", "manual": "PyTorch manual", "torch_nn": "torch.nn"}

In [ ]:
with open(f"{ART}/metrics.json") as f:
    info = json.load(f)
COST_CALL = info["costs"]["call"]
COST_CHURN = info["costs"]["churn"]

prep = joblib.load(f"{ART}/preprocessing.pkl")
test = pd.read_csv(f"{ART}/test_predictions.csv")
y_true = test["y_true"].values

# Model 1: scikit-learn
sk_model = joblib.load(f"{ART}/sklearn_model.pkl")

# Model 2: manual PyTorch (just weights + bias)
manual = torch.load(f"{ART}/manual_pytorch.pt")

# Model 3: torch.nn — same class definition as in train, then load the weights
class LogReg(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)

nn_model = LogReg(len(prep["feature_cols"]))
nn_model.load_state_dict(torch.load(f"{ART}/torch_nn.pt"))
nn_model.eval()

In [ ]:
raw_cols = [c for c in test.columns if c not in ["y_true", "p_sklearn", "p_manual", "p_torch_nn"]]
X = pd.get_dummies(test[raw_cols], drop_first=True)
X = X.reindex(columns=prep["feature_cols"], fill_value=0)
X[prep["num_cols"]] = prep["scaler"].transform(X[prep["num_cols"]])
X = X.astype("float32").values

with torch.no_grad():
    Xt = torch.tensor(X)
    probs = {
        "sklearn": sk_model.predict_proba(X)[:, 1],
        "manual": torch.sigmoid(Xt @ manual["w"] + manual["b"]).numpy().ravel(),
        "torch_nn": torch.sigmoid(nn_model(Xt)).numpy().ravel(),
    }

# Sanity check: the loaded models reproduce the predictions saved during training
for key, col in [("sklearn", "p_sklearn"), ("manual", "p_manual"), ("torch_nn", "p_torch_nn")]:
    print(key, "max diff vs saved:", float(np.abs(probs[key] - test[col].values).max()))

In [ ]:
def confusion(y, proba, threshold):
    pred = (proba >= threshold).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    return tp, fp, fn, tn


def cost_at(y, proba, threshold):
    tp, fp, fn, tn = confusion(y, proba, threshold)
    return fp * COST_CALL + fn * COST_CHURN


THRESHOLDS = np.round(np.arange(0.01, 1.00, 0.01), 2)
NEVER_CALL = int(y_true.sum()) * COST_CHURN
CALL_ALL = int((y_true == 0).sum()) * COST_CALL


def metrics_table():
    rows = []
    for m in info["metrics"]:
        rows.append({
            "Model": m["model"],
            "ROC-AUC": m["roc_auc"],
            "Accuracy @0.5": m["accuracy@0.5"],
            "Cost @0.5 ($)": m["cost@0.5"],
        })
    b = info["baselines"]
    rows.append({"Model": "Baseline: never call", "ROC-AUC": None,
                 "Accuracy @0.5": b["majority_class_accuracy"], "Cost @0.5 ($)": b["never_call_cost"]})
    rows.append({"Model": "Baseline: call everyone", "ROC-AUC": None,
                 "Accuracy @0.5": None, "Cost @0.5 ($)": b["call_everyone_cost"]})
    return pd.DataFrame(rows)


def pred_vs_actual(model_key):
    df = pd.DataFrame({
        "Predicted churn probability": probs[model_key],
        "Actual": np.where(y_true == 1, "Churned", "Stayed"),
    })
    fig = px.histogram(
        df, x="Predicted churn probability", color="Actual", nbins=40,
        barmode="overlay", opacity=0.65,
        title=f"{MODEL_LABELS[model_key]}: predicted probability vs. actual outcome",
    )
    fig.update_layout(yaxis_title="Customers in test set")
    return fig


def agreement_plot():
    df = pd.DataFrame({
        "scikit-learn": probs["sklearn"],
        "PyTorch manual": probs["manual"],
        "torch.nn": probs["torch_nn"],
    })
    long = df.melt(id_vars="scikit-learn", var_name="PyTorch model", value_name="PyTorch probability")
    fig = px.scatter(long, x="scikit-learn", y="PyTorch probability", color="PyTorch model",
                     opacity=0.5, title="Do the three models agree? (each dot = one test customer)")
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Perfect agreement",
                             line=dict(dash="dash", color="gray")))
    fig.update_layout(xaxis_title="scikit-learn probability")
    return fig


def feature_plot(feature):
    df = test[[feature]].copy()
    df["Churn"] = np.where(y_true == 1, "Churned", "Stayed")
    if pd.api.types.is_numeric_dtype(df[feature]) and df[feature].nunique() > 10:
        fig = px.histogram(df, x=feature, color="Churn", nbins=30, barmode="overlay", opacity=0.65)
    else:
        df[feature] = df[feature].astype(str)
        rate = df.groupby(feature)["Churn"].apply(lambda s: (s == "Churned").mean()).reset_index()
        rate.columns = [feature, "Churn rate"]
        fig = px.bar(rate, x=feature, y="Churn rate", text_auto=".0%")
        fig.update_yaxes(tickformat=".0%")
    fig.update_layout(title=f"{feature} (test set)")
    return fig


def target_plot():
    counts = pd.Series(np.where(y_true == 1, "Churned", "Stayed")).value_counts().reset_index()
    counts.columns = ["Outcome", "Customers"]
    return px.bar(counts, x="Outcome", y="Customers", text_auto=True,
                  title=f"Target distribution (test set, churn rate {y_true.mean():.1%})")


def threshold_view(model_key, threshold):
    proba = probs[model_key]
    tp, fp, fn, tn = confusion(y_true, proba, threshold)
    cost = fp * COST_CALL + fn * COST_CHURN

    cm = go.Figure(go.Heatmap(
        z=[[tn, fp], [fn, tp]],
        x=["Predicted: stay", "Predicted: churn"],
        y=["Actual: stayed", "Actual: churned"],
        text=[[f"TN<br>{tn}", f"FP<br>{fp}"], [f"FN<br>{fn}", f"TP<br>{tp}"]],
        texttemplate="%{text}", colorscale="Blues", showscale=False,
    ))
    cm.update_layout(title=f"Confusion matrix @ threshold {threshold:.2f}", yaxis_autorange="reversed")

    costs = [cost_at(y_true, proba, t) for t in THRESHOLDS]
    best_t = float(THRESHOLDS[int(np.argmin(costs))])
    curve = go.Figure()
    curve.add_trace(go.Scatter(x=THRESHOLDS, y=costs, mode="lines", name="Model cost"))
    curve.add_hline(y=NEVER_CALL, line_dash="dot", annotation_text="Never call")
    curve.add_hline(y=CALL_ALL, line_dash="dash", annotation_text="Call everyone")
    curve.add_vline(x=threshold, line_color="red", annotation_text="Current")
    curve.update_layout(title="Business cost vs. threshold", xaxis_title="Threshold", yaxis_title="Total cost ($)")

    calls = tp + fp
    summary = (
        f"### Business cost: ${cost:,}\n"
        f"- Customers called: **{calls}** ({fp} of them would have stayed → {fp} × ${COST_CALL} = ${fp * COST_CALL:,})\n"
        f"- Churners missed: **{fn}** → {fn} × ${COST_CHURN} = ${fn * COST_CHURN:,}\n"
        f"- Churners caught: **{tp}** of {tp + fn}\n\n"
        f"Baselines: never call = ${NEVER_CALL:,} · call everyone = ${CALL_ALL:,}  \n"
        f"Cheapest threshold for this model: **{best_t:.2f}** (${min(costs):,})"
    )
    return cm, curve, summary

In [ ]:
model_choices = [(label, key) for key, label in MODEL_LABELS.items()]
feature_choices = [c for c in raw_cols]

with gr.Blocks(title="Telco Churn — retention dashboard") as demo:
    gr.Markdown(
        "# Telco Churn — retention dashboard\n"
        f"Logistic regression trained three ways. Costs: **${COST_CALL}** per retention call, "
        f"**${COST_CHURN}** per lost customer. All numbers are on the held-out test set "
        f"({len(y_true)} customers)."
    )

    with gr.Tab("Model comparison"):
        gr.Dataframe(metrics_table(), label="Test-set metrics")
        cmp_model = gr.Dropdown(model_choices, value="sklearn", label="Model")
        cmp_plot = gr.Plot(pred_vs_actual("sklearn"))
        cmp_model.change(pred_vs_actual, cmp_model, cmp_plot)
        gr.Plot(agreement_plot())

    with gr.Tab("Distributions"):
        gr.Plot(target_plot())
        feat = gr.Dropdown(feature_choices, value="tenure_group", label="Feature")
        feat_plot = gr.Plot(feature_plot("tenure_group"))
        feat.change(feature_plot, feat, feat_plot)

    with gr.Tab("Threshold & business cost"):
        with gr.Row():
            th_model = gr.Dropdown(model_choices, value="sklearn", label="Model")
            th_slider = gr.Slider(0.01, 0.99, value=0.5, step=0.01, label="Decision threshold")
        th_text = gr.Markdown()
        with gr.Row():
            th_cm = gr.Plot()
            th_curve = gr.Plot()
        outputs = [th_cm, th_curve, th_text]
        th_model.change(threshold_view, [th_model, th_slider], outputs)
        th_slider.change(threshold_view, [th_model, th_slider], outputs)
        demo.load(threshold_view, [th_model, th_slider], outputs)

demo.launch()